# Reporte de calidad Bronze → Silver v4 (Scopus y OpenAlex)

Este notebook es el reporte reproducible que demuestra que Silver v4 está listo para construir la capa
Gold. Lee los mismos directorios de Bronze y Silver que usa el contenedor programado y es de solo
lectura: no llama a ninguna API, no accede a credenciales y no escribe en `data/`. Conviene montar
`data/` en modo de solo lectura al ejecutarlo.

El corpus se congeló el 2026-10-09; los valores de este reporte describen ese corte.

## Contenido

1. Metadatos de lotes por fuente y estado (puntos de control de la ingesta).
2. Páginas Parquet crudas de Bronze por fuente.
3. Planificación de ventanas de Scopus.
4. Resumen de estados de Bronze: lotes terminales para Silver y lotes con páginas que no lo son.
5. Estado de procesamiento Spark por lote.
6. Totales de Silver v4 por fuente (solo la versión publicada de cada grupo de lotes).
7. Cobertura por año de publicación × fuente: resumen, tópicos y autores.
8. Indicadores de calidad por año × fuente: `matches_common_query`, `is_eligible` y `work_type_norm`.
9. Rechazos por `rejection_reason` × fuente.
10. Fracción muestreada por año × fuente frente a los totales de la API.
11. Solapamiento entre fuentes por DOI normalizado.
12. Limitaciones.

Las zonas vacías se informan explícitamente, por lo que el notebook también puede ejecutarse antes del
primer lote programado.

In [1]:
from pathlib import Path
import json
import sys

from pyspark.sql import SparkSession, functions as F

BRONZE = Path("/data/bronze")
SILVER = Path("/data/silver")
PROJECT_ROOT = Path("/app")
if PROJECT_ROOT.exists() and str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.compaction import active_rejected_paths, active_silver_paths
from src.spark_pipeline import SILVER_SCHEMA_VERSION, TERMINAL_STATUSES, unique_publications

spark = (
    SparkSession.builder.master("local[*]")
    .appName("openalex-pipeline-evidence")
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")


def pct(condition):
    """Percentage of rows where a boolean condition holds, rounded to one decimal."""
    return F.round(100 * F.avg(F.coalesce(condition, F.lit(False)).cast("int")), 1)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/10 11:41:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 1. Metadatos de lotes

Cada lote de Bronze deja un JSON en `batch_metadata/`; es el índice de auditoría de la ingesta.

In [2]:
# 1. Batch metadata is the audit index for ingestion and processing state.
metadata_rows = []
for path in sorted((BRONZE / "batch_metadata").glob("*.json")):
    item = json.loads(path.read_text(encoding="utf-8"))
    metadata_rows.append({
        "source": str(item.get("source") or "unknown"),
        "batch_id": str(item.get("batch_id")),
        "mode": str(item.get("mode") or "unknown"),
        "status": str(item.get("status") or "unknown"),
        "stop_reason": str(item.get("stop_reason") or ""),
        "record_count": int(item.get("record_count", 0) or 0),
        "page_count": int(item.get("page_count", 0) or 0),
        "total_results": int(item.get("total_results", 0) or 0),
        "created_at": str(item.get("created_at") or "unknown"),
    })
batch_metadata_df = spark.createDataFrame(metadata_rows) if metadata_rows else None
if batch_metadata_df is not None:
    print(f"Batch metadata rows: {batch_metadata_df.count():,}")
    (
        batch_metadata_df.groupBy("source", "status")
        .agg(F.count("*").alias("batches"), F.sum("record_count").alias("records"), F.sum("page_count").alias("pages"))
        .orderBy("source", "status")
        .show(truncate=False)
    )
else:
    print("No batch metadata has been written yet.")

Batch metadata rows: 5,293


+--------+---------------------+-------+-------+-----+
|source  |status               |batches|records|pages|
+--------+---------------------+-------+-------+-----+
|openalex|completed_target     |100    |604000 |3028 |
|openalex|frozen               |13     |39200  |196  |
|openalex|quota_exceeded       |4712   |44000  |220  |
|scopus  |completed_exhausted  |188    |404250 |16238|
|scopus  |completed_target     |20     |41832  |1682 |
|scopus  |failed               |1      |850    |34   |
|scopus  |frozen               |1      |900    |36   |
|scopus  |oversized            |257    |0      |0    |
|scopus  |result_window_reached|1      |3403   |137  |
+--------+---------------------+-------+-------+-----+



## 2. Páginas crudas de Bronze

Se cuentan los archivos `page-*.parquet`, independientemente de los archivos auxiliares de Spark.

In [3]:
# 2. Raw pages are Parquet files; count them independently from auxiliary Spark files.
raw_page_counts = [
    {"source": source, "raw_page_files": len(list((BRONZE / source).glob("*/page-*.parquet")))}
    for source in ("openalex", "scopus")
]
spark.createDataFrame(raw_page_counts).orderBy("source").show(truncate=False)
if not any(row["raw_page_files"] for row in raw_page_counts):
    print("No raw Bronze pages have been written yet.")

+--------------+--------+
|raw_page_files|source  |
+--------------+--------+
|3444          |openalex|
|18127         |scopus  |
+--------------+--------+



## 3. Ventanas de Scopus

Las ventanas que superan el límite de 5000 resultados se dividen (`oversized`) hasta que cada una cabe,
por lo que los primeros ciclos pueden tener muchas ventanas sin registros. `completed_exhausted` indica
una ventana leída por completo y `frozen` una ventana pausada que se congeló al detener la ingesta.

In [4]:
# 3. Scopus windows: 'oversized' was split further, 'completed_exhausted' was fully read,
# 'frozen' was paused mid-plan and frozen when ingestion stopped for good.
scopus_df = batch_metadata_df.filter(F.col("source") == "scopus") if batch_metadata_df is not None else None
if scopus_df is not None and scopus_df.count():
    scopus_df.groupBy("status", "stop_reason").agg(
        F.count("*").alias("windows"), F.sum("record_count").alias("records")
    ).orderBy(F.desc("windows")).show(truncate=False)
    print(f"Oversized windows already split: {scopus_df.filter(F.col('status') == 'oversized').count()}")
    scopus_df.filter(F.col("record_count") > 0).select(
        "batch_id", "status", "record_count", "total_results"
    ).orderBy("created_at").show(20, truncate=False)
else:
    print("No Scopus windows have been planned yet.")

+---------------------+---------------------+-------+-------+
|status               |stop_reason          |windows|records|
+---------------------+---------------------+-------+-------+
|oversized            |oversized            |257    |0      |
|completed_exhausted  |completed_exhausted  |188    |404250 |
|completed_target     |completed_target     |20     |41832  |
|failed               |failed               |1      |850    |
|result_window_reached|result_window_reached|1      |3403   |
|frozen               |time_budget_reached  |1      |900    |
+---------------------+---------------------+-------+-------+



Oversized windows already split: 257


+--------------------------------------+-------------------+------------+-------------+
|batch_id                              |status             |record_count|total_results|
+--------------------------------------+-------------------+------------+-------------+
|scopus-20261003T013040271626Z-b93bec6b|completed_target   |1597        |19118        |
|scopus-20261003T014045747306Z-c7d2a303|completed_target   |1664        |3980         |
|scopus-20261003T014834118784Z-cba45df6|completed_exhausted|39          |39           |
|scopus-20261003T014844076765Z-aae7b9ef|completed_target   |1675        |3322         |
|scopus-20261003T015445140964Z-b6645132|completed_exhausted|65          |65           |
|scopus-20261003T015450261007Z-3b47a73a|completed_exhausted|69          |69           |
|scopus-20261003T015458848543Z-21373f8b|completed_target   |1570        |1868         |
|scopus-20261003T020042658502Z-3e024186|completed_exhausted|1599        |1599         |
|scopus-20261003T020508350869Z-9

## 4. Estados de Bronze y lotes terminales para Silver

Silver procesa solo los lotes con páginas cuyo estado es terminal para su fuente
(`TERMINAL_STATUSES` en `src/spark_pipeline.py`): nunca se reanudarán, así que sus páginas son
definitivas. Tras congelar los lotes pausados, **no debe quedar ningún lote con páginas que no sea
terminal**; si los hay, sus registros todavía no están en Silver.

In [5]:
# 4. Bronze status summary: which batches hold pages and whether Silver treats them as terminal.
completed_only = TERMINAL_STATUSES.get("openalex", set()) & TERMINAL_STATUSES.get("scopus", set())
for item in metadata_rows:
    terminal = TERMINAL_STATUSES.get(item["source"], completed_only)
    item["terminal_for_silver"] = item["status"] in terminal
    item["holds_pages"] = item["page_count"] > 0

for source, statuses in sorted(TERMINAL_STATUSES.items()):
    print(f"Terminal statuses for {source}: {sorted(statuses)}")

if metadata_rows:
    status_df = spark.createDataFrame(metadata_rows)
    (
        status_df.groupBy("source", "status", "terminal_for_silver", "holds_pages")
        .agg(F.count("*").alias("batches"), F.sum("record_count").alias("records"))
        .orderBy("source", "status", "holds_pages")
        .show(50, truncate=False)
    )
    stranded = [row for row in metadata_rows if row["holds_pages"] and not row["terminal_for_silver"]]
    print(f"Batches that hold pages but are NOT terminal for Silver: {len(stranded)}")
    print(f"Records in those batches: {sum(row['record_count'] for row in stranded):,}")
    for row in stranded[:20]:
        print(f"  {row['source']} {row['batch_id']} status={row['status']} records={row['record_count']}")
else:
    print("No batch metadata has been written yet.")

Terminal statuses for openalex: ['completed', 'completed_exhausted', 'completed_target', 'failed', 'frozen', 'quota_exceeded']
Terminal statuses for scopus: ['completed', 'completed_exhausted', 'completed_target', 'failed', 'frozen', 'result_window_reached']


+--------+---------------------+-------------------+-----------+-------+-------+
|source  |status               |terminal_for_silver|holds_pages|batches|records|
+--------+---------------------+-------------------+-----------+-------+-------+
|openalex|completed_target     |true               |true       |100    |604000 |
|openalex|frozen               |true               |true       |13     |39200  |
|openalex|quota_exceeded       |true               |false      |4698   |0      |
|openalex|quota_exceeded       |true               |true       |14     |44000  |
|scopus  |completed_exhausted  |true               |false      |35     |0      |
|scopus  |completed_exhausted  |true               |true       |153    |404250 |
|scopus  |completed_target     |true               |true       |20     |41832  |
|scopus  |failed               |true               |true       |1      |850    |
|scopus  |frozen               |true               |true       |1      |900    |
|scopus  |oversized         

## 5. Estado de procesamiento Spark

Para cada lote terminal con páginas se lee su estado en `silver/_processing/`. Todos deben figurar como
`completed`; `-1` indica que el valor todavía no está disponible.

In [6]:
# 5. Processing state of every batch Silver must process (terminal and holding pages).
processing_rows = []
for item in metadata_rows:
    if not (item["holds_pages"] and item["terminal_for_silver"]):
        continue
    state_path = SILVER / "_processing" / f"{item['batch_id']}.json"
    state = json.loads(state_path.read_text(encoding="utf-8")) if state_path.exists() else {}
    processing_rows.append({
        "source": item["source"],
        "batch_id": item["batch_id"],
        "ingestion_status": item["status"],
        "processing_status": str(state.get("status", "not_processed")),
        "valid_rows": int(state.get("valid_rows", -1)),
        "rejected_rows": int(state.get("rejected_rows", -1)),
        "write_seconds": float(state.get("write_seconds", -1.0)),
    })
if processing_rows:
    processing_df = spark.createDataFrame(processing_rows)
    done = sum(row["processing_status"] == "completed" for row in processing_rows)
    print(f"Terminal batches with pages: {len(processing_rows):,}; processed: {done:,}; pending: {len(processing_rows) - done:,}")
    (
        processing_df.groupBy("source", "ingestion_status", "processing_status")
        .agg(
            F.count("*").alias("batches"),
            F.sum(F.greatest("valid_rows", F.lit(0))).alias("valid_rows"),
            F.sum(F.greatest("rejected_rows", F.lit(0))).alias("rejected_rows"),
            F.round(F.sum(F.greatest("write_seconds", F.lit(0.0))), 1).alias("write_seconds"),
        )
        .orderBy("source", "ingestion_status", "processing_status")
        .show(50, truncate=False)
    )
    processing_df.filter(F.col("processing_status") != "completed").orderBy("source", "batch_id").show(20, truncate=False)
else:
    print("No terminal ingestion batches with data are available to process yet.")

Terminal batches with pages: 303; processed: 303; pending: 0


+--------+---------------------+-----------------+-------+----------+-------------+-------------+
|source  |ingestion_status     |processing_status|batches|valid_rows|rejected_rows|write_seconds|
+--------+---------------------+-----------------+-------+----------+-------------+-------------+
|openalex|completed_target     |completed        |100    |482225    |121775       |2956.5       |
|openalex|frozen               |completed        |13     |31685     |7515         |477.2        |
|openalex|quota_exceeded       |completed        |14     |43936     |64           |203.8        |
|scopus  |completed_exhausted  |completed        |153    |402431    |1819         |3043.1       |
|scopus  |completed_target     |completed        |20     |41752     |80           |210.1        |
|scopus  |failed               |completed        |1      |850       |0            |23.0         |
|scopus  |frozen               |completed        |1      |895       |5            |38.2         |
|scopus  |result_win

+--------+----------------+-----------------+-------------+------+----------+-------------+
|batch_id|ingestion_status|processing_status|rejected_rows|source|valid_rows|write_seconds|
+--------+----------------+-----------------+-------------+------+----------+-------------+
+--------+----------------+-----------------+-------------+------+----------+-------------+



## 6. Totales de Silver v4

Silver se lee siempre a través de `active_silver_paths` (`src/compaction.py`), que resuelve solo la
versión publicada de cada grupo de lotes según el manifiesto de compactación. Leer `works/` directamente
contaría dos veces los lotes ya compactados.

In [7]:
# 6. Silver reads only the published version of each batch group (compaction manifest).
schema_file = SILVER / "_schema_version.json"
on_disk_version = json.loads(schema_file.read_text(encoding="utf-8")).get("version") if schema_file.exists() else None
print(f"Silver schema version on disk: {on_disk_version}; code version: {SILVER_SCHEMA_VERSION}")

silver_paths = active_silver_paths(SILVER)
silver_df = None
if silver_paths:
    silver_df = spark.read.option("mergeSchema", "true").parquet(*(str(path) for path in silver_paths)).cache()
    print(f"Active Silver paths: {len(silver_paths):,}")
    print(f"Silver rows read: {silver_df.count():,}")
    silver_df.printSchema()
    silver_df.groupBy("source", "schema_version").count().orderBy("source", "schema_version").show()
    duplicates = silver_df.groupBy("source", "source_id").count().filter(F.col("count") > 1).count()
    print(f"Duplicate (source, source_id) pairs: {duplicates:,}")
    unique_df = unique_publications(silver_df)
    print(f"DOI-deduplicated analysis-view rows: {unique_df.count():,}")
    unique_df.groupBy("source").count().orderBy("source").show()
else:
    print("No published Silver Parquet batches have been written yet.")

Silver schema version on disk: 4; code version: 4


Active Silver paths: 140


Silver rows read: 1,007,148
root
 |-- batch_id: string (nullable = true)
 |-- source: string (nullable = true)
 |-- source_id: string (nullable = true)
 |-- doi: string (nullable = true)
 |-- normalized_doi: string (nullable = true)
 |-- ingested_at: string (nullable = true)
 |-- schema_version: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- publication_year: integer (nullable = true)
 |-- publication_date: string (nullable = true)
 |-- work_type: string (nullable = true)
 |-- language: string (nullable = true)
 |-- venue: string (nullable = true)
 |-- abstract: string (nullable = true)
 |-- keywords: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- has_abstract: boolean (nullable = true)
 |-- topics: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- id: string (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- score: double (nullable = true)
 |    |    |-- subfield: string

+--------+--------------+------+
|  source|schema_version| count|
+--------+--------------+------+
|openalex|             4|557846|
|  scopus|             4|449302|
+--------+--------------+------+



Duplicate (source, source_id) pairs: 0


DOI-deduplicated analysis-view rows: 964,401


+--------+------+
|  source| count|
+--------+------+
|openalex|557484|
|  scopus|406917|
+--------+------+



In [8]:
# 6b. Overall profile per source. Scopus STANDARD view has no abstract, topics or author list;
# its country arrives as a name (country_names) instead of an ISO code (country_codes).
if silver_df is not None:
    silver_df.groupBy("source").agg(
        F.count("*").alias("rows"),
        pct(F.col("has_abstract")).alias("pct_abstract"),
        pct(F.col("has_topics")).alias("pct_topics"),
        pct(F.col("author_count") > 0).alias("pct_authors"),
        pct((F.size("country_codes") > 0) | (F.size("country_names") > 0)).alias("pct_country"),
        F.round(F.avg("cited_by_count"), 2).alias("avg_citations"),
        F.min("publication_year").alias("min_year"),
        F.max("publication_year").alias("max_year"),
    ).orderBy("source").show(truncate=False)
else:
    print("Silver profile skipped: no Silver data yet.")

+--------+------+------------+----------+-----------+-----------+-------------+--------+--------+
|source  |rows  |pct_abstract|pct_topics|pct_authors|pct_country|avg_citations|min_year|max_year|
+--------+------+------------+----------+-----------+-----------+-------------+--------+--------+
|openalex|557846|75.9        |98.0      |98.5       |72.6       |36.12        |2011    |2026    |
|scopus  |449302|0.0         |0.0       |0.0        |98.6       |28.83        |2011    |2026    |
+--------+------+------------+----------+-----------+-----------+-------------+--------+--------+



## 7. Cobertura por año × fuente

Porcentaje de filas con resumen, con tópicos y con al menos un autor. La vista STANDARD de Scopus no
entrega resumen, tópicos ni la lista de autores, por lo que en Scopus esas coberturas son nulas por
diseño (`src/silver_columns.py`). Todo análisis por tópico o autoría se construye con OpenAlex.

In [9]:
# 7. Coverage per publication_year x source.
if silver_df is not None:
    (
        silver_df.groupBy("publication_year", "source")
        .agg(
            F.count("*").alias("rows"),
            pct(F.col("has_abstract")).alias("pct_abstract"),
            pct(F.col("has_topics")).alias("pct_topics"),
            pct(F.col("author_count") > 0).alias("pct_authors"),
        )
        .orderBy("publication_year", "source")
        .show(100, truncate=False)
    )
else:
    print("Coverage skipped: no Silver data yet.")

+----------------+--------+------+------------+----------+-----------+
|publication_year|source  |rows  |pct_abstract|pct_topics|pct_authors|
+----------------+--------+------+------------+----------+-----------+
|2011            |openalex|6901  |65.8        |99.2      |97.8       |
|2011            |scopus  |7550  |0.0         |0.0       |0.0        |
|2012            |openalex|7116  |63.8        |99.6      |97.8       |
|2012            |scopus  |7646  |0.0         |0.0       |0.0        |
|2013            |openalex|8582  |67.6        |99.6      |98.4       |
|2013            |scopus  |7574  |0.0         |0.0       |0.0        |
|2014            |openalex|8717  |66.8        |99.6      |98.3       |
|2014            |scopus  |9404  |0.0         |0.0       |0.0        |
|2015            |openalex|8287  |68.0        |99.5      |98.3       |
|2015            |scopus  |8702  |0.0         |0.0       |0.0        |
|2016            |openalex|9183  |69.6        |99.5      |98.3       |
|2016 

## 8. Indicadores de calidad de Silver v4

- `matches_common_query`: el registro coincide con la consulta común a ambas fuentes (en Scopus es
  siempre verdadero).
- `is_eligible`: el registro es elegible para el análisis según el tipo de trabajo normalizado y las
  reglas de `src/silver_rules.py`.
- `work_type_norm`: tipo de trabajo normalizado entre fuentes.

Silver conserva todas las filas; Gold decide qué filtrar a partir de estos indicadores.

In [10]:
# 8. Flags per source and per publication_year x source.
if silver_df is not None:
    silver_df.groupBy("source").agg(
        F.count("*").alias("rows"),
        pct(F.col("matches_common_query")).alias("pct_common_query"),
        pct(F.col("is_eligible")).alias("pct_eligible"),
        F.sum(F.col("is_eligible").cast("int")).alias("eligible_rows"),
    ).orderBy("source").show(truncate=False)
    (
        silver_df.groupBy("publication_year", "source")
        .agg(
            F.count("*").alias("rows"),
            pct(F.col("matches_common_query")).alias("pct_common_query"),
            pct(F.col("is_eligible")).alias("pct_eligible"),
        )
        .orderBy("publication_year", "source")
        .show(100, truncate=False)
    )
else:
    print("Flags skipped: no Silver data yet.")

+--------+------+----------------+------------+-------------+
|source  |rows  |pct_common_query|pct_eligible|eligible_rows|
+--------+------+----------------+------------+-------------+
|openalex|557846|98.9            |74.1        |413264       |
|scopus  |449302|100.0           |94.2        |423100       |
+--------+------+----------------+------------+-------------+



+----------------+--------+------+----------------+------------+
|publication_year|source  |rows  |pct_common_query|pct_eligible|
+----------------+--------+------+----------------+------------+
|2011            |openalex|6901  |92.5            |81.3        |
|2011            |scopus  |7550  |100.0           |96.9        |
|2012            |openalex|7116  |91.4            |82.5        |
|2012            |scopus  |7646  |100.0           |95.4        |
|2013            |openalex|8582  |92.7            |83.7        |
|2013            |scopus  |7574  |100.0           |95.1        |
|2014            |openalex|8717  |93.6            |82.8        |
|2014            |scopus  |9404  |100.0           |96.0        |
|2015            |openalex|8287  |96.0            |79.5        |
|2015            |scopus  |8702  |100.0           |96.7        |
|2016            |openalex|9183  |97.8            |76.4        |
|2016            |scopus  |9824  |100.0           |95.0        |
|2017            |openale

In [11]:
# 8b. work_type_norm distribution per source, and per year as a pivot of row counts.
if silver_df is not None:
    source_totals = silver_df.groupBy("source").agg(F.count("*").alias("source_rows"))
    (
        silver_df.groupBy("source", "work_type_norm", "is_eligible")
        .count()
        .join(source_totals, "source")
        .withColumn("pct_of_source", F.round(100 * F.col("count") / F.col("source_rows"), 1))
        .drop("source_rows")
        .orderBy("source", F.desc("count"))
        .show(60, truncate=False)
    )
    for source in ("openalex", "scopus"):
        print(f"work_type_norm rows per publication_year: {source}")
        (
            silver_df.filter(F.col("source") == source)
            .groupBy("publication_year")
            .pivot("work_type_norm")
            .count()
            .na.fill(0)
            .orderBy("publication_year")
            .show(100, truncate=False)
        )
else:
    print("work_type_norm distribution skipped: no Silver data yet.")

+--------+----------------+-----------+------+-------------+
|source  |work_type_norm  |is_eligible|count |pct_of_source|
+--------+----------------+-----------+------+-------------+
|openalex|article         |true       |304126|54.5         |
|openalex|conference_paper|true       |99932 |17.9         |
|openalex|preprint        |false      |61623 |11.0         |
|openalex|other           |false      |38407 |6.9          |
|openalex|book_chapter    |false      |33386 |6.0          |
|openalex|review          |true       |9206  |1.7          |
|openalex|book            |false      |7465  |1.3          |
|openalex|editorial       |false      |2216  |0.4          |
|openalex|article         |false      |664   |0.1          |
|openalex|erratum         |false      |435   |0.1          |
|openalex|letter          |false      |205   |0.0          |
|openalex|conference_paper|false      |174   |0.0          |
|openalex|review          |false      |7     |0.0          |
|scopus  |article       

+----------------+-------+----+------------+----------------+---------+-------+------+-----+--------+------+
|publication_year|article|book|book_chapter|conference_paper|editorial|erratum|letter|other|preprint|review|
+----------------+-------+----+------------+----------------+---------+-------+------+-----+--------+------+
|2011            |3961   |145 |561         |1636            |20       |3      |1     |484  |73      |17    |
|2012            |4225   |142 |438         |1613            |14       |3      |0     |523  |120     |38    |
|2013            |5291   |136 |417         |1859            |19       |7      |0     |595  |224     |34    |
|2014            |5065   |138 |440         |2118            |24       |2      |0     |643  |244     |43    |
|2015            |4212   |114 |353         |2342            |29       |5      |1     |689  |508     |34    |
|2016            |4182   |118 |406         |2826            |31       |6      |0     |767  |836     |11    |
|2017            |5

+----------------+-------+----+------------+----------------+---------+-------+------+-----+------+
|publication_year|article|book|book_chapter|conference_paper|editorial|erratum|letter|other|review|
+----------------+-------+----+------------+----------------+---------+-------+------+-----+------+
|2011            |2080   |1   |6           |5136            |103      |9      |6     |110  |99    |
|2012            |4660   |1   |7           |2330            |131      |11     |48    |153  |305   |
|2013            |4798   |3   |36          |2166            |131      |22     |30    |147  |241   |
|2014            |6977   |5   |9           |1749            |96       |14     |48    |203  |303   |
|2015            |6218   |1   |14          |1903            |83       |15     |38    |133  |297   |
|2016            |7550   |6   |217         |1448            |79       |15     |46    |129  |334   |
|2017            |9126   |4   |242         |3255            |97       |26     |64    |219  |440   |


## 9. Rechazos por motivo y fuente

Se leen los grupos compactados y los lotes aún no compactados mediante `active_rejected_paths`. La
fuente se obtiene de los metadatos del lote y, si faltan, del prefijo `scopus-` del `batch_id`.

In [12]:
# 9. Rejection reasons x source, reading compacted groups plus batches not yet compacted.
rejected_paths = [path for path in active_rejected_paths(SILVER) if path.exists()]
if rejected_paths:
    rejected_df = spark.read.parquet(*(str(path) for path in rejected_paths))
    batch_sources = spark.createDataFrame(
        [(row["batch_id"], row["source"]) for row in metadata_rows] or [("", "")], "batch_id string, meta_source string"
    )
    rejected_df = rejected_df.join(F.broadcast(batch_sources), "batch_id", "left").withColumn(
        "source",
        F.coalesce(
            F.col("meta_source"),
            F.when(F.col("batch_id").startswith("scopus-"), "scopus").otherwise("openalex"),
        ),
    )
    print(f"Rejected rows: {rejected_df.count():,}")
    rejected_df.groupBy("source", "rejection_reason").count().orderBy("source", F.desc("count")).show(truncate=False)
    if silver_df is not None:
        valid_counts = silver_df.groupBy("source").agg(F.count("*").alias("valid_rows"))
        (
            rejected_df.groupBy("source").agg(F.count("*").alias("rejected_rows"))
            .join(valid_counts, "source", "outer")
            .na.fill(0)
            .withColumn(
                "pct_rejected",
                F.round(100 * F.col("rejected_rows") / (F.col("rejected_rows") + F.col("valid_rows")), 1),
            )
            .orderBy("source")
            .show(truncate=False)
        )
else:
    print("No rejected rows have been written yet.")

Rejected rows: 131,287


+--------+----------------------------------+------+
|source  |rejection_reason                  |count |
+--------+----------------------------------+------+
|openalex|duplicate_source_id_previous_batch|114629|
|openalex|duplicate_source_id               |14725 |
|scopus  |duplicate_source_id               |1076  |
|scopus  |duplicate_source_id_previous_batch|855   |
|scopus  |missing_title                     |2     |
+--------+----------------------------------+------+



+--------+-------------+----------+------------+
|source  |rejected_rows|valid_rows|pct_rejected|
+--------+-------------+----------+------------+
|openalex|129354       |557846    |18.8        |
|scopus  |1933         |449302    |0.4         |
+--------+-------------+----------+------------+



## 10. Fracción muestreada por año × fuente

Filas de Silver dividido por el total que la API informó para cada año al planificar la descarga
(`data/bronze/plans/{openalex,scopus}.json`, campo `plan.totals`). El año de Silver es
`publication_year` del registro y el del plan es el filtro de la consulta, por lo que pueden diferir en
unos pocos registros. Gold debe trabajar con proporciones dentro de cada año, no con conteos absolutos.

In [13]:
# 10. Sampling fraction per year x source: Silver rows / API totals from the Bronze plans.
plan_rows = []
for source in ("openalex", "scopus"):
    plan_path = BRONZE / "plans" / f"{source}.json"
    if not plan_path.exists():
        print(f"No plan file for {source}.")
        continue
    totals = json.loads(plan_path.read_text(encoding="utf-8")).get("plan", {}).get("totals", {})
    plan_rows.extend({"source": source, "publication_year": int(year), "api_total": int(total)} for year, total in totals.items())

if plan_rows and silver_df is not None:
    plan_df = spark.createDataFrame(plan_rows)
    silver_year_df = silver_df.groupBy("source", "publication_year").agg(
        F.count("*").alias("silver_rows"),
        F.sum(F.col("is_eligible").cast("int")).alias("eligible_rows"),
    )
    fraction_df = (
        plan_df.join(silver_year_df, ["source", "publication_year"], "full_outer")
        .na.fill(0, ["silver_rows", "eligible_rows"])
        .withColumn("pct_sampled", F.round(100 * F.col("silver_rows") / F.col("api_total"), 2))
        .withColumn("pct_sampled_eligible", F.round(100 * F.col("eligible_rows") / F.col("api_total"), 2))
    )
    fraction_df.orderBy("publication_year", "source").show(100, truncate=False)
    (
        fraction_df.groupBy("source")
        .agg(F.sum("api_total").alias("api_total"), F.sum("silver_rows").alias("silver_rows"))
        .withColumn("pct_sampled", F.round(100 * F.col("silver_rows") / F.col("api_total"), 2))
        .orderBy("source")
        .show(truncate=False)
    )
else:
    print("Sampling fraction skipped: plans or Silver data missing.")

+--------+----------------+---------+-----------+-------------+-----------+--------------------+
|source  |publication_year|api_total|silver_rows|eligible_rows|pct_sampled|pct_sampled_eligible|
+--------+----------------+---------+-----------+-------------+-----------+--------------------+
|openalex|2011            |80735    |6901       |5609         |8.55       |6.95                |
|scopus  |2011            |57993    |7550       |7315         |13.02      |12.61               |
|openalex|2012            |84463    |7116       |5872         |8.42       |6.95                |
|scopus  |2012            |60451    |7646       |7295         |12.65      |12.07               |
|openalex|2013            |89973    |8582       |7182         |9.54       |7.98                |
|scopus  |2013            |62251    |7574       |7205         |12.17      |11.57               |
|openalex|2014            |93864    |8717       |7221         |9.29       |7.69                |
|scopus  |2014            |618

+--------+---------+-----------+-----------+
|source  |api_total|silver_rows|pct_sampled|
+--------+---------+-----------+-----------+
|openalex|6229309  |557846     |8.96       |
|scopus  |3632371  |449302     |12.37      |
+--------+---------+-----------+-----------+



## 11. Solapamiento entre fuentes

Número de DOI normalizados presentes en ambas fuentes. La vista de análisis deduplicada
(`unique_publications`) conserva una sola fila por DOI.

In [14]:
# 11. Cross-source overlap on normalized DOI.
if silver_df is not None:
    doi_df = (
        silver_df.filter(F.col("normalized_doi").isNotNull() & (F.col("normalized_doi") != ""))
        .select("source", "normalized_doi")
        .distinct()
    )
    doi_df.groupBy("source").agg(F.count("*").alias("distinct_dois")).orderBy("source").show()
    shared = doi_df.groupBy("normalized_doi").agg(F.countDistinct("source").alias("sources")).filter(F.col("sources") == 2).count()
    print(f"Normalized DOIs present in both sources: {shared:,}")
    silver_df.groupBy("source").agg(
        pct(F.col("normalized_doi").isNotNull() & (F.col("normalized_doi") != "")).alias("pct_with_doi")
    ).orderBy("source").show()
else:
    print("Overlap skipped: no Silver data yet.")

+--------+-------------+
|  source|distinct_dois|
+--------+-------------+
|openalex|       514576|
|  scopus|       441061|
+--------+-------------+



Normalized DOIs present in both sources: 42,310


+--------+------------+
|  source|pct_with_doi|
+--------+------------+
|openalex|        92.3|
|  scopus|        98.2|
+--------+------------+



## 12. Limitaciones

Este reporte prueba que Silver v4 es completo y coherente con Bronze, pero no corrige los sesgos del
muestreo. En particular:

- La muestra de cada año es un top por relevancia (OpenAlex) o por fecha de carga (Scopus), no una
  muestra aleatoria.
- La fracción muestreada difiere entre años y fuentes; 2026 es un año parcial.
- Las fuentes no buscan exactamente la misma población (`matches_common_query`).
- Los tópicos solo existen en OpenAlex.

El detalle y sus consecuencias para Gold están en
[Limitaciones del corpus y del muestreo](../documentacion/6.limitaciones-muestreo.md).

In [15]:
spark.stop()